In [1]:
# Cell 2: Import libraries and setup
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, GatedGraphConv, BatchNorm, LayerNorm
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import zipfile
import pickle
import math
import torch_geometric

# Initialize CUDA context explicitly
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"CUDA initialized. Available GPUs: {torch.cuda.device_count()}")
else:
    print("CUDA not available, using CPU")

# Print torch_geometric version
print(f"torch_geometric version: {torch_geometric.__version__}")

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device for multi-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
else:
    print("Using 1 GPU or CPU")

CUDA initialized. Available GPUs: 2
torch_geometric version: 2.6.1
Using 2 GPUs!


In [2]:
# Cell 2: Extract dataset
zip_path = 'METR-LA.zip'
data_dir = 'METR-LA'
if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
print(f"Data directory {data_dir} ready")

Data directory METR-LA ready


In [3]:
# Cell 3: Load adjacency matrix
def load_adjacency_matrix(adj_matrix_path, data_dir='METR-LA'):
    try:
        with open(adj_matrix_path, 'rb') as f:
            adj_matrix = pickle.load(f)
        
        if isinstance(adj_matrix, tuple):
            for i in range(len(adj_matrix)):
                temp_matrix = np.array(adj_matrix[i])
                if temp_matrix.ndim == 2 and np.issubdtype(temp_matrix.dtype, np.number):
                    adj_matrix = temp_matrix
                    break
            else:
                raise ValueError(f"No 2D numeric array found in tuple: {adj_matrix}")
        elif isinstance(adj_matrix, dict):
            possible_keys = ['adj', 'adj_mx', 'matrix', 'adj_matrix', 'connectivity']
            for key in possible_keys:
                if key in adj_matrix:
                    adj_matrix = adj_matrix[key]
                    break
            else:
                raise ValueError(f"Adjacency matrix not found in dict. Keys found: {list(adj_matrix.keys())}")
        elif isinstance(adj_matrix, list):
            adj_matrix = np.array(adj_matrix)
        elif not isinstance(adj_matrix, np.ndarray):
            raise ValueError(f"Unexpected type for adjacency matrix: {type(adj_matrix)}")
        
        adj_matrix = np.array(adj_matrix, dtype=np.float32)
        adj_matrix = (adj_matrix + adj_matrix.T) / 2  # Symmetrize
        
        if adj_matrix.ndim != 2:
            raise ValueError(f"Expected 2D adjacency matrix, got shape {adj_matrix.shape}")
        if adj_matrix.shape[0] != adj_matrix.shape[1]:
            raise ValueError(f"Expected square adjacency matrix, got shape {adj_matrix.shape}")
        if adj_matrix.shape[0] != 207:
            raise ValueError(f"Expected 207x207 adjacency matrix for METR-LA, got shape {adj_matrix.shape}")
        
        if not np.issubdtype(adj_matrix.dtype, np.number):
            raise ValueError(f"Adjacency matrix contains non-numeric values, dtype: {adj_matrix.dtype}")
        
        non_zero_count = np.count_nonzero(adj_matrix)
        print(f"Number of non-zero entries in adjacency matrix: {non_zero_count}")
        if non_zero_count == 0:
            raise ValueError("Adjacency matrix has no non-zero entries, cannot create edge_index")
        
        print(f"Loaded adjacency matrix from {adj_matrix_path}, shape: {adj_matrix.shape}, dtype: {adj_matrix.dtype}")
        return adj_matrix
    
    except Exception as e:
        print(f"Warning: Failed to load adjacency matrix from {adj_matrix_path}: {str(e)}")
        print(f"Falling back to default adjacency matrix: {data_dir}/adj_mat.npy")
        adj_matrix = np.load(os.path.join(data_dir, 'adj_mat.npy'))
        adj_matrix = (adj_matrix + adj_matrix.T) / 2  # Symmetrize
        non_zero_count = np.count_nonzero(adj_matrix)
        print(f"Loaded default adjacency matrix from {data_dir}/adj_mat.npy, shape: {adj_matrix.shape}, dtype: {adj_matrix.dtype}, non-zero entries: {non_zero_count}")
        return adj_matrix

In [4]:
# Cell 4: Load METR-LA data
def load_metr_la_data(data_dir='METR-LA', num_days=99, custom_adj_path=None):
    if custom_adj_path is not None:
        print(f"Loading custom adjacency matrix from: {custom_adj_path}")
        adj_mat = load_adjacency_matrix(custom_adj_path, data_dir)
    else:
        adj_mat = np.load(os.path.join(data_dir, 'adj_mat.npy'))
        adj_mat = (adj_mat + adj_mat.T) / 2  # Symmetrize
        non_zero_count = np.count_nonzero(adj_mat)
        print(f"Loaded default adjacency matrix from {data_dir}/adj_mat.npy, non-zero entries: {non_zero_count}")
    
    print(f"Adjacency matrix shape: {adj_mat.shape}, dtype: {adj_mat.dtype}")

    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))
    print(f"Original node_values shape: {node_values.shape}")

    time_steps = num_days * 288
    node_values = node_values[:time_steps]
    if node_values.ndim == 3:
        node_values = node_values[:, :, 0]
        node_values = node_values.T
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")

    print(f"Processed node_values shape: {node_values.shape}")

    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T

    edge_index = np.where(adj_mat > 0)
    print(f"Raw edge_index from np.where: {len(edge_index)} arrays, shapes: {[arr.shape for arr in edge_index]}")
    if len(edge_index) != 2:
        raise ValueError(f"Expected edge_index to have 2 arrays (source, target), got {len(edge_index)}")
    edge_index = np.stack(edge_index, axis=0)
    print(f"Stacked edge_index shape: {edge_index.shape}")
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    if edge_index.shape[0] != 2:
        raise ValueError(f"Expected edge_index to have shape [2, num_edges], got shape {edge_index.shape}")
    if edge_index.shape[1] == 0:
        raise ValueError("edge_index is empty; adjacency matrix may have no non-zero entries")
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)

    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)

    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")

    return node_values, edge_index, edge_weight, scaler

In [5]:
# Cell 5: Create data samples and 100th day data
def add_time_features(features, current_time_step_index, num_intervals_per_day=288, in_timestep=12):
    batch_size, current_in_timestep, num_nodes = features.shape
    time_features_list = []
    for i in range(current_in_timestep):
        global_time_step = (current_time_step_index + i) % num_intervals_per_day
        sin_time = math.sin(2 * math.pi * global_time_step / num_intervals_per_day)
        cos_time = math.cos(2 * math.pi * global_time_step / num_intervals_per_day)
        time_features = torch.tensor([[sin_time, cos_time]] * num_nodes, dtype=torch.float32)
        time_features_list.append(time_features)
    time_features_tensor = torch.stack(time_features_list, dim=0)
    time_features_batch = time_features_tensor.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    features_reshaped = features.unsqueeze(-1)
    features_with_time = torch.cat((features_reshaped, time_features_batch), dim=-1)
    return features_with_time

def create_data_samples(node_values, in_timestep=12, out_timestep=12, stride=1):
    features_list, labels_list = [], []
    num_nodes, total_time_steps = node_values.shape
    time_step_indices = range(0, total_time_steps - in_timestep - out_timestep + 1, stride)
    
    for i in time_step_indices:
        speed_features = node_values[:, i:i + in_timestep]
        labels = node_values[:, i + in_timestep:i + in_timestep + out_timestep]
        features_list.append(speed_features)
        labels_list.append(labels)
    
    features = torch.tensor(np.array(features_list, dtype=np.float32)).permute(0, 2, 1)
    labels = torch.tensor(np.array(labels_list, dtype=np.float32)).permute(0, 2, 1)
    
    features_with_time_list = [
        add_time_features(
            features[i].unsqueeze(0),
            global_start_time_step,
            num_intervals_per_day=288,
            in_timestep=in_timestep
        ).squeeze(0)
        for i, global_start_time_step in enumerate(time_step_indices)
    ]
    
    features_with_time = torch.stack(features_with_time_list, dim=0)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)
    return features_final, labels

def create_100th_day_data(data_dir='METR-LA', in_timestep=12, out_timestep=12, stride=1):
    node_values_full = np.load(os.path.join(data_dir, 'node_values.npy'))
    start_global_time_index = 288 * 99
    node_values = node_values_full[start_global_time_index : start_global_time_index + 288]
    node_values = node_values[:, :, 0].T
    scaler_100 = MinMaxScaler()
    node_values_scaled = scaler_100.fit_transform(node_values.T).T
    num_nodes, total_time_steps_100 = node_values_scaled.shape
    
    time_step_indices = range(0, total_time_steps_100 - in_timestep - out_timestep + 1, stride)
    features_list, labels_list = [], []
    
    for i in time_step_indices:
        speed_features = node_values_scaled[:, i:i + in_timestep]
        labels = node_values_scaled[:, i + in_timestep:i + in_timestep + out_timestep]
        features_list.append(speed_features)
        labels_list.append(labels)
    
    features = torch.tensor(np.array(features_list, dtype=np.float32)).permute(0, 2, 1)
    labels = torch.tensor(np.array(labels_list, dtype=np.float32)).permute(0, 2, 1)
    
    features_with_time_list = [
        add_time_features(
            features[i].unsqueeze(0),
            start_global_time_index + local_start_time_step,
            num_intervals_per_day=288,
            in_timestep=in_timestep
        ).squeeze(0)
        for i, local_start_time_step in enumerate(time_step_indices)
    ]
    
    features_with_time = torch.stack(features_with_time_list, dim=0)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)
    return features_final, labels, scaler_100

In [6]:
# Cell 6: Compute metrics and visualize predictions
def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape

def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction.png'):
    os.makedirs(output_dir, exist_ok=True)
    batch_size, num_nodes, out_timestep = pred.shape
    pred_reshaped = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_reshaped = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_reshaped).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_reshaped).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_to_plot = pred_unscaled[0].T
    target_to_plot = target_unscaled[0].T
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred_to_plot.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred_to_plot[i], label='Predicted', marker='o', markersize=3)
        plt.plot(target_to_plot[i], label='Actual', marker='x', markersize=3)
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (Sample)')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

def visualize_full_100th_day(day_100_loader, model, scaler, output_dir='plots', num_sensors=5, filename='full_100th_day_prediction.png'):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    num_nodes = all_preds.shape[1]
    full_preds_accum = np.zeros((num_nodes, 288))
    full_targets_accum = np.zeros((num_nodes, 288))
    counts = np.zeros((num_nodes, 288))
    for i in range(all_preds.shape[0]):
        start_idx_in_day = i + 12
        end_idx_in_day = start_idx_in_day + 12
        full_preds_accum[:, start_idx_in_day:end_idx_in_day] += all_preds[i]
        full_targets_accum[:, start_idx_in_day:end_idx_in_day] += all_targets[i]
        counts[:, start_idx_in_day:end_idx_in_day] += 1
    mask = counts > 0
    full_preds_mean = np.zeros_like(full_preds_accum)
    full_targets_mean = np.zeros_like(full_targets_accum)
    full_preds_mean[mask] = full_preds_accum[mask] / counts[mask]
    full_targets_mean[mask] = full_targets_accum[mask] / counts[mask]
    full_preds_unscaled = scaler.inverse_transform(full_preds_mean.T).T
    full_targets_unscaled = scaler.inverse_transform(full_targets_mean.T).T
    covered_time_steps = slice(12, 288 - 12)
    time_steps = np.arange(12, 288 - 12) * 5 / 60
    os.makedirs(output_dir, exist_ok=True)
    plt.figure(figsize=(15, 10))
    for i in range(min(num_sensors, num_nodes)):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(time_steps, full_preds_unscaled[i, covered_time_steps], label='Predicted', marker='o', markersize=3)
        plt.plot(time_steps, full_targets_unscaled[i, covered_time_steps], label='Actual', marker='x', markersize=3)
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day, Hours {time_steps[0]:.0f}-{time_steps[-1]:.0f})')
        plt.xlabel('Time (hours)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

In [7]:
# Cell 7: Define Graph Transformer model
class GraphTransformerLayer(nn.Module):
    def __init__(self, hidden_channels, heads=8, dropout=0.1):
        super(GraphTransformerLayer, self).__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(
            hidden_channels, hidden_channels, heads=heads, concat=False, dropout=dropout
        )
        self.norm1 = LayerNorm(hidden_channels)
        self.norm2 = LayerNorm(hidden_channels)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight, batch):
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_gcn = self.norm1(x_gcn)
        x_trans = self.transformer(x_gcn, edge_index)
        x = x_gcn + x_trans + x
        x = self.norm2(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x_ffn = self.ffn(x)
        x = x + x_ffn
        return x

class GraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=12, num_layers=3, pe_dim=20, heads=4, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim if use_pe else hidden_channels)
        if use_pe:
            self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.layers = nn.ModuleList([GraphTransformerLayer(hidden_channels, heads=heads, dropout=0.1) for _ in range(num_layers)])
        self.norm = LayerNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels // 2, out_channels)
        )
    
    def forward(self, x, edge_index, edge_weight, pe=None, batch=None):
        x = self.node_emb(x)
        if self.use_pe and pe is not None:
            pe = self.pe_lin(pe)
            x = torch.cat([x, pe], dim=-1)
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
        x = self.norm(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1 if batch is not None else 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

In [8]:
# Cell 8: Main workflow
adj_matrices = [
    'adj_mx_knn_lane.pkl',
    'adj_mx_knn.pkl',
    'adj_mx_radius_lane.pkl',
    'adj_mx_radius.pkl'
]
stride_options = {
    '1': ('5 minutes', 1),
    '2': ('1 hour', 12),
    '3': ('1 day', 288)
}
results = {}

for adj_file in adj_matrices:
    for stride_key, (stride_name, stride_val) in stride_options.items():
        print(f"\n--- Processing Adjacency Matrix: {adj_file}, Stride: {stride_name} ({stride_val} time steps) ---")
        
        try:
            # Load data
            node_values, edge_index, edge_weight, scaler = load_metr_la_data(custom_adj_path=f'METR-LA/{adj_file}')
            in_timestep = 12
            out_timestep = 12

            # Create data samples with specified stride
            features, labels = create_data_samples(node_values, in_timestep=in_timestep, out_timestep=out_timestep, stride=stride_val)
            
            # Split data
            total_samples = len(features)
            train_end = int(0.6 * total_samples)
            val_end = train_end + int(0.2 * total_samples)
            
            train_features = features[:train_end]
            train_labels = labels[:train_end]
            val_features = features[train_end:val_end]
            val_labels = labels[train_end:val_end]
            test_features = features[val_end:]
            test_labels = labels[val_end:]
            
            print(f"Train samples: {len(train_features)}, Val samples: {len(val_features)}, Test samples: {len(test_features)}")
            
            # Apply random walk positional encoding
            transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
            
            # Create datasets
            train_data = [transform(Data(x=f, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(train_features, train_labels)]
            val_data = [transform(Data(x=f, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(val_features, val_labels)]
            test_data = [transform(Data(x=f, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(test_features, test_labels)]
            
            # Create 100th day data with stride=1
            day_100_features, day_100_labels, day_100_scaler = create_100th_day_data(in_timestep=in_timestep, out_timestep=out_timestep, stride=1)
            day_100_data = [transform(Data(x=f, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(day_100_features, day_100_labels)]
            
            # Create data loaders
            train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
            test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
            day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)
            
            # Initialize model
            model_in_channels = in_timestep * 3  # 1 speed + 2 time features
            model = GraphTransformer(in_channels=model_in_channels, hidden_channels=64, out_channels=out_timestep, num_layers=4, pe_dim=20, heads=4, use_pe=True).to(device)
            
            # Optimizer and loss
            optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
            criterion = nn.MSELoss()
            
            # Training loop
            for epoch in range(20):
                model.train()
                total_train_loss = 0
                for data in train_loader:
                    data = data.to(device)
                    optimizer.zero_grad()
                    out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
                    target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                    loss = criterion(out, target)
                    loss.backward()
                    optimizer.step()
                    total_train_loss += loss.item()
                train_loss = total_train_loss / len(train_loader)
                
                model.eval()
                total_val_loss = 0
                with torch.no_grad():
                    for data in val_loader:
                        data = data.to(device)
                        out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
                        target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                        loss = criterion(out, target)
                        total_val_loss += loss.item()
                val_loss = total_val_loss / len(val_loader)
                print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            
            # Evaluate on test set
            model.eval()
            total_mae, total_rmse, total_mape = 0, 0, 0
            num_batches = 0
            with torch.no_grad():
                for data in test_loader:
                    data = data.to(device)
                    out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
                    target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                    mae, rmse, mape = compute_metrics(out, target, scaler)
                    total_mae += mae
                    total_rmse += rmse
                    total_mape += mape
                    num_batches += 1
                    if num_batches == 1:
                        visualize_predictions(out, target, scaler, output_dir=f'plots_test_{stride_val}', filename=f'prediction_test_{adj_file}_stride_{stride_val}.png')
                test_mae = total_mae / num_batches
                test_rmse = total_rmse / num_batches
                test_mape = total_mape / num_batches
                print(f"Test MAE: {test_mae:.4f} mph, RMSE: {test_rmse:.4f} mph, MAPE: {test_mape:.4f}%")
            
            # Evaluate on 100th day
            total_mae, total_rmse, total_mape = 0, 0, 0
            num_batches = 0
            with torch.no_grad():
                for data in day_100_loader:
                    data = data.to(device)
                    out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
                    target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                    mae, rmse, mape = compute_metrics(out, target, day_100_scaler)
                    total_mae += mae
                    total_rmse += rmse
                    total_mape += mape
                    num_batches += 1
                    if num_batches == 1:
                        visualize_predictions(out, target, day_100_scaler, output_dir=f'plots_100th_{stride_val}', filename=f'prediction_100th_day_{adj_file}_stride_{stride_val}.png')
                day_100_mae = total_mae / num_batches
                day_100_rmse = total_rmse / num_batches
                day_100_mape = total_mape / num_batches
                print(f"100th Day MAE: {day_100_mae:.4f} mph, RMSE: {day_100_rmse:.4f} mph, MAPE: {day_100_mape:.4f}%")
            
            # Visualize full 100th day
            visualize_full_100th_day(day_100_loader, model, day_100_scaler, output_dir=f'plots_100th_{stride_val}', filename=f'full_100th_day_prediction_{adj_file}_stride_{stride_val}.png')
            
            # Store results
            results[f"{adj_file}_stride_{stride_val}"] = {
                'test_mae': test_mae, 'test_rmse': test_rmse, 'test_mape': test_mape,
                'day_100_mae': day_100_mae, 'day_100_rmse': day_100_rmse, 'day_100_mape': day_100_mape
            }
        
        except Exception as e:
            print(f"Error processing {adj_file} with stride {stride_val}: {str(e)}")
            results[f"{adj_file}_stride_{stride_val}"] = {
                'test_mae': float('nan'), 'test_rmse': float('nan'), 'test_mape': float('nan'),
                'day_100_mae': float('nan'), 'day_100_rmse': float('nan'), 'day_100_mape': float('nan')
            }

# Print summary of results
print("\nSummary of Results:")
for key, metrics in results.items():
    adj_file_name, stride_info = key.split('_stride_')
    print(f"{adj_file_name}, Stride {stride_info}:")
    print(f"  Test - MAE: {metrics['test_mae']:.4f} mph, RMSE: {metrics['test_rmse']:.4f} mph, MAPE: {metrics['test_mape']:.4f}%")
    print(f"  100th Day - MAE: {metrics['day_100_mae']:.4f} mph, RMSE: {metrics['day_100_rmse']:.4f} mph, MAPE: {metrics['day_100_mape']:.4f}%")


--- Processing Adjacency Matrix: adj_mx_knn_lane.pkl, Stride: 5 minutes (1 time steps) ---
Loading custom adjacency matrix from: METR-LA/adj_mx_knn_lane.pkl
Number of non-zero entries in adjacency matrix: 742
Loaded adjacency matrix from METR-LA/adj_mx_knn_lane.pkl, shape: (207, 207), dtype: float32
Adjacency matrix shape: (207, 207), dtype: float32
Original node_values shape: (34272, 207, 2)
Processed node_values shape: (207, 28512)
Raw edge_index from np.where: 2 arrays, shapes: [(742,), (742,)]
Stacked edge_index shape: (2, 742)
edge_index device: cuda:0, shape: torch.Size([2, 742])
edge_weight device: cuda:0, shape: torch.Size([742])
Train samples: 17093, Val samples: 5697, Test samples: 5699
Epoch 1, Train Loss: 0.0540, Val Loss: 0.0253
Epoch 2, Train Loss: 0.0321, Val Loss: 0.0213
Epoch 3, Train Loss: 0.0293, Val Loss: 0.0218
Epoch 4, Train Loss: 0.0277, Val Loss: 0.0206
Epoch 5, Train Loss: 0.0268, Val Loss: 0.0204
Epoch 6, Train Loss: 0.0262, Val Loss: 0.0203
Epoch 7, Train Lo